In [ ]:
import pandas as pd
import numpy as np
import gc
from IPython.display import display
from mnp.ingestion.loader import load_endes
from mnp.utils.profiler import centrar_notebook
from mnp.utils.cleaning import run_phase1_engine
# Configuración visual
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
centrar_notebook()

print("[OK]")

In [ ]:
# Carga de datos
history = load_endes(
        year=range(2007, 2025),
        module="housing",
        record="rech23h",
        meta=True
    )

dfs = []
col_labels_hist = {}
val_labels_hist = {}

for year in sorted(list(history.keys())):
    year_df, year_meta = history.pop(year)
    col_labels_hist[year] = year_meta.column_names_to_labels
    val_labels_hist[year] = year_meta.variable_value_labels
    
    dfs.append(year_df.assign(year=year))

df_raw = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f"[OK] Data RECH23 cargada: {len(df_raw)} filas y {df_raw.shape[1]} columnas.")
print(f"[OK] Diccionario cargado: {len(col_labels_hist)} Column Labels, {len(val_labels_hist)} Value Labels")

In [ ]:
# Inspeccionar variable
# df_raw['HC51'].value_counts(dropna=False)
df_raw.head(10)

In [ ]:
from mnp.utils.cleaning import run_phase1_engine
from mnp.configs.rech23 import config_f1

# 3. Limpiar
df_clean = run_phase1_engine(df_raw, config_f1)

print(f'[OK] Limpieza biológica completa. Shape: {df_clean.shape}')
df_clean.head(5)

In [ ]:
from importlib import reload
import mnp.utils.cleaning
reload(mnp.utils.cleaning)
from mnp.utils.cleaning import apply_standard_labels
from mnp.configs.rech23 import config_f1, config_f3
# Le pasamos config_f1 para que el motor sepa mágicamente dónde buscar 
# gracias a tus propias reglas de Coalesce.
df_final = apply_standard_labels(df_clean, val_labels_hist, config_f3, config_f1)
print(f"[OK] Fase 3 (Estandarización de Etiquetas) completada.")
df_final.head(15)

In [ ]:
from mnp.config import INTERIM_DATA_DIR
import os
# Aseguramos que la carpeta interim exista
os.makedirs(INTERIM_DATA_DIR, exist_ok=True)
# Ruta del archivo final
output_path = INTERIM_DATA_DIR / "rech23_cleaned.parquet"
# Guardamos en formato parquet (preserva tipos de datos y ahorra RAM)
df_final.to_parquet(output_path, index=False)
print(f"[OK] Datos RECH23 (Fase 1, 2 y 3) guardados exitosamente en:")
print(f"-> {output_path}")

In [ ]:
import sweetviz as sv
from mnp.configs.column_labels import SWEETVIZ_LABELS
from mnp.config import INTERIM_DATA_DIR  # Importamos tu variable mágica

df_sweetviz = df_final.rename(columns=SWEETVIZ_LABELS)
reporte_sv = sv.analyze(df_sweetviz)

# Usamos la ruta absoluta inteligente y la pasamos a string
ruta_reporte = str(INTERIM_DATA_DIR / 'rech23_sweetviz_report.html')

reporte_sv.show_html(ruta_reporte)
reporte_sv.show_notebook(w="100%", h="800", filepath=ruta_reporte)


In [ ]:
df_final['HV270'].value_counts(dropna=False)